# Multinomial Naive Bayes (Multinomial Event Model)

## Learning Objectives
- Understand the **multinomial event model** and how it differs from the Bernoulli event model
- Derive the MAP decision rule for **word-count** feature vectors
- Compute **MLE estimates** for $\phi_{kj}$ with Laplace smoothing
- Implement a vectorised `fit / predict_log_proba / predict` in NumPy
- Recognise why **absent words do not contribute** to the score (key contrast with Bernoulli NB)

## Problem Statement

Given a labelled training set $\{(x^{(i)}, y^{(i)})\}_{i=1}^{m}$ with **count** features $x^{(i)} \in \mathbb{Z}_{\geq 0}^n$ (e.g. word frequencies) and class labels $y^{(i)} \in \{0, \ldots, K{-}1\}$, learn a classifier that assigns the most probable class to a new count vector $x$.

---

### Motivating Example — Spam Classification

Represent each e-mail as a **bag-of-words count vector** over a vocabulary $V$:

| Feature | Value | Meaning |
|---|---|---|
| `x_"money"` | 3 | word *money* appears 3 times |
| `x_"free"` | 2 | word *free* appears twice |
| `x_"meeting"` | 0 | word *meeting* absent |

Goal: classify as **spam** ($y=1$) or **ham** ($y=0$).

---

### Event Model Comparison

| Model | Feature | What it captures | Absent words |
|---|---|---|---|
| **Bernoulli event model** | $x_j \in \{0,1\}$ | Word *presence* | Penalised via $(1-\phi_{kj})$ |
| **Multinomial event model** ← *this notebook* | $x_j \in \mathbb{Z}_{\geq 0}$ | Word *frequency* | Ignored (zero contribution) |

## The Multinomial Event Model Assumption

A document of length $N = \sum_j x_j$ is modelled as $N$ words drawn **i.i.d.** from a class-specific multinomial distribution over the vocabulary.

The probability of observing count vector $x$ given class $k$ is:

$$P(x \mid y=k) = \binom{N}{x_1, x_2, \ldots, x_n} \prod_{j=1}^{n} \phi_{kj}^{x_j}$$

where $\phi_{kj} = P(\text{word}_j \mid y=k)$ and $\sum_j \phi_{kj} = 1$.

**For classification the multinomial coefficient can be dropped** — it does not depend on $k$ and cancels in the argmax:

$$\hat{y} = \arg\max_{k}\; P(y=k) \prod_{j=1}^{n} \phi_{kj}^{x_j}$$

> **Why absent words don't matter:** If $x_j = 0$ then $\phi_{kj}^0 = 1$, contributing nothing to the product. In contrast, the Bernoulli model still multiplies $(1-\phi_{kj})$ for absent words.

## Model / Hypothesis

The MAP decision rule in **log-space**:

$$\hat{y} = \arg\max_{k} \left[ \log P(y=k) + \sum_{j=1}^{n} x_j \log \phi_{kj} \right]$$

This is a **linear** function of the count vector $x$, enabling efficient matrix-multiply scoring:

$$\text{score}_k = \log \phi_k + x^\top \log \boldsymbol{\phi}_k$$

where $\boldsymbol{\phi}_k = [\phi_{k1}, \ldots, \phi_{kn}]^\top \in \Delta^{n-1}$ (probability simplex).

## Derivation

**Step 1 — Bayes' theorem.**

$$P(y=k \mid x) = \frac{P(x \mid y=k)\, P(y=k)}{P(x)}$$

**Step 2 — Drop evidence and multinomial coefficient.** Both are constant w.r.t. $k$:

$$\hat{y} = \arg\max_{k}\; P(y=k) \prod_{j=1}^{n} \phi_{kj}^{x_j}$$

**Step 3 — Take log.**

$$\hat{y} = \arg\max_{k} \left[ \log P(y=k) + \sum_{j=1}^{n} x_j \log \phi_{kj} \right]$$

**Step 4 — MLE parameter estimates.**

$$\phi_k = \frac{m_k}{m}, \qquad \phi_{kj} = \frac{\sum_{i:\,y^{(i)}=k} x^{(i)}_j}{\sum_{j'} \sum_{i:\,y^{(i)}=k} x^{(i)}_{j'}}$$

i.e. the fraction of all words in class-$k$ documents that are word $j$.

**Step 5 — Laplace smoothing** (avoids $\log 0$ for unseen words):

$$\phi_{kj} = \frac{\left(\sum_{i:\,y^{(i)}=k} x^{(i)}_j\right) + \alpha}{\left(\sum_{j'} \sum_{i:\,y^{(i)}=k} x^{(i)}_{j'}\right) + \alpha V}$$

where $V = n$ is the vocabulary size. The denominator adds $\alpha$ pseudo-counts for **each** of the $V$ words, keeping $\sum_j \phi_{kj} = 1$.

## Algorithm

**Training — `fit(X, y, alpha=1)`**

1. Find unique classes $\{0, \ldots, K{-}1\}$
2. For each class $k$, collect $X_k = X[y == k]$
3. Compute log prior: $\log\phi_k = \log(m_k / m)$
4. Sum word counts per class: $c_k = \text{sum}(X_k,\;\text{axis}=0)$ — shape $(n)$
5. Compute smoothed word probabilities: $\phi_{kj} = (c_{kj} + \alpha)\;/\;(\sum_j c_{kj} + \alpha V)$ — shape $(n)$
6. Store $\log \boldsymbol{\phi}_k$ — shape $(K, n)$

**Prediction — `predict(X, params)`**

1. $\text{log\_scores} = \log\phi + X \cdot (\log\boldsymbol{\phi})^\top$ — shape $(m, K)$
2. Return $\hat{y} = \arg\max_k\; \text{log\_scores}$

---

### Shape Reference

| Symbol | Shape | Description |
|---|---|---|
| $X$ | $(m, n)$ | Word-count feature matrix |
| $y$ | $(m)$ | Integer class labels |
| `log_prior` | $(K)$ | Log class priors |
| `log_phi` | $(K, n)$ | $\log P(\text{word}_j \mid y=k)$ — smoothed |
| `log_scores` | $(m, K)$ | Log posterior score per sample per class |

In [ ]:
import numpy as np


def fit(X, y, alpha=1.0):
    """
    Fit Multinomial Naive Bayes (Multinomial Event Model).

    Inputs
    ------
    X     : ndarray, (m, n)  non-negative integer count matrix
    y     : ndarray, (m,)    integer class labels 0 .. K-1
    alpha : float            Laplace smoothing pseudo-count (default 1)

    Output
    ------
    params : dict
        log_prior : ndarray (K,)    log P(y=k)
        log_phi   : ndarray (K, n)  log P(word_j | y=k)  — Laplace-smoothed
    """
    classes = np.unique(y)
    K = len(classes)
    m, n = X.shape

    log_prior = np.zeros(K)
    log_phi   = np.zeros((K, n))

    for k in classes:
        Xk = X[y == k]                                 # (m_k, n)
        log_prior[k] = np.log(len(Xk) / m)
        counts_k = Xk.sum(axis=0)                      # (n,)  total word counts in class k
        # Laplace-smoothed probabilities; denominator adds alpha for each of the V words
        phi_k     = (counts_k + alpha) / (counts_k.sum() + alpha * n)
        log_phi[k] = np.log(phi_k)

    return dict(log_prior=log_prior, log_phi=log_phi)


def predict_log_proba(X, params):
    """
    Compute log posterior score for each class.

    Inputs
    ------
    X      : ndarray, (m, n)  count test features
    params : dict              output of fit()

    Output
    ------
    log_scores : ndarray, (m, K)  unnormalised log posterior per class
    """
    lp      = params['log_prior']  # (K,)
    log_phi = params['log_phi']    # (K, n)
    # score_k = log_prior_k + x @ log_phi[k]
    # Vectorised: X (m,n) @ log_phi.T (n,K) -> (m, K)
    return lp + X @ log_phi.T     # (m, K)


def predict(X, params):
    """
    Predict class labels for test samples.

    Inputs
    ------
    X      : ndarray, (m, n)  count test features
    params : dict              output of fit()

    Output
    ------
    y_hat : ndarray, (m,)  predicted class labels
    """
    return np.argmax(predict_log_proba(X, params), axis=1)

## Key Properties

| Property | Detail |
|---|---|
| **Type** | Generative classifier — models $P(x, y)$ |
| **Feature type** | Non-negative integers (word counts, TF, etc.) |
| **Training** | $O(m \cdot n)$ — one pass per class |
| **Prediction** | $O(K \cdot n)$ per sample (single matrix multiply) |
| **Absent words** | Contribute $x_j \log\phi_{kj} = 0$ — **no penalty** |
| **Laplace smoothing** | $\alpha=1$, denominator $\sum_j c_{kj} + \alpha V$ (one pseudo-count per vocab word) |
| **Underflow guard** | All likelihoods computed in log-space |
| **Typical use** | Text classification with bag-of-words counts or TF-IDF |

---

### Side-by-Side: Bernoulli vs Multinomial Event Model

| | Bernoulli | Multinomial |
|---|---|---|
| Feature | $x_j \in \{0,1\}$ | $x_j \in \mathbb{Z}_{\geq 0}$ |
| Per-feature likelihood | $\phi_{kj}^{x_j}(1-\phi_{kj})^{1-x_j}$ | $\phi_{kj}^{x_j}$ |
| Absent word contribution | $(1-\phi_{kj})$ — penalised | $1$ — ignored |
| Laplace denom. | $m_k + 2\alpha$ | $\sum_j c_{kj} + \alpha V$ |
| Score formula | $\log\phi_k + x^\top\log\boldsymbol{\phi}_k + (1-x)^\top\log(1-\boldsymbol{\phi}_k)$ | $\log\phi_k + x^\top\log\boldsymbol{\phi}_k$ |